# IngExuity conversation-aware QLoRA training

Run this notebook on a Kaggle GPU session. Attach your conversation JSONL as a Kaggle input dataset and add a Kaggle secret named `HF_TOKEN` with access to `meta-llama/Llama-3.2-1B-Instruct`.


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/kaggle/working/ingexuity')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'feat/conversation-sft-pipeline', '--single-branch',
        'https://github.com/toxzak-svg/ingexuity.git', str(REPO)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'python/requirements.txt')
], check=True)
print('Repository and dependencies are ready.')


In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets.')
except Exception as exc:
    raise RuntimeError(
        'Add a Kaggle secret named HF_TOKEN, then rerun this cell. '
        'The token must have access to meta-llama/Llama-3.2-1B-Instruct.'
    ) from exc


In [ ]:
candidates = sorted(
    list(Path('/kaggle/input').rglob('*.jsonl')) +
    list(Path('/kaggle/input').rglob('*.json')),
    key=lambda path: path.stat().st_size,
    reverse=True,
)
if not candidates:
    raise FileNotFoundError('No .jsonl or .json file was found under /kaggle/input.')

for index, path in enumerate(candidates[:20]):
    print(f'{index}: {path} ({path.stat().st_size / 1_048_576:.2f} MiB)')

DATA_PATH = candidates[0]
print(f'\nSelected dataset: {DATA_PATH}')


In [ ]:
subprocess.run([
    sys.executable, str(REPO / 'python/training_data.py'),
    '--data', str(DATA_PATH), '--summary-only'
], check=True)


## Smoke test

This trains one epoch on 50 normalized assistant-turn examples. Run this before the full job.


In [ ]:
SMOKE_OUTPUT = Path('/kaggle/working/ingexuity-smoke')
subprocess.run([
    sys.executable, str(REPO / 'python/train_llama_simple.py'),
    '--data', str(DATA_PATH),
    '--output', str(SMOKE_OUTPUT),
    '--smoke-test', '50',
    '--epochs', '1',
    '--batch', '1',
    '--grad-accum', '4',
    '--maxlen', '1024',
], check=True)
print(f'Smoke-test adapter saved under: {SMOKE_OUTPUT}')


## Full training

Run this cell only after the smoke test succeeds.


In [ ]:
FULL_OUTPUT = Path('/kaggle/working/ingexuity-full')
subprocess.run([
    sys.executable, str(REPO / 'python/train_llama_simple.py'),
    '--data', str(DATA_PATH),
    '--output', str(FULL_OUTPUT),
    '--epochs', '3',
    '--batch', '2',
    '--grad-accum', '8',
    '--maxlen', '1024',
], check=True)
print(f'Full adapter saved under: {FULL_OUTPUT}')


In [ ]:
archive_path = '/kaggle/working/ingexuity-training-results.tar.gz'
subprocess.run([
    'tar', '-czf', archive_path,
    '-C', '/kaggle/working',
    'ingexuity-smoke', 'ingexuity-full'
], check=True)
print(archive_path)
